# HPC Egocentric Step 3 Local Analysis (SVG Only)

This notebook reruns **Step 3** locally for all four run folders under `data/egocentric_tuning_carpenter`.

- Runs: `head_all_spike`, `head_simple_spike`, `head_complex_spike`, `travel_all_spike`
- Full recompute: existing `per_cell_summary`, `summary_stats`, and `egocentric_tuning_summary.csv` are removed first
- Output format: `svg` only


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

BASE_DIR = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4')
NOTEBOOKS_HPC = BASE_DIR / 'notebooks_HPC'
DATA_ROOT = BASE_DIR / 'data'
FIGURES_ROOT = BASE_DIR / 'figures'
MANIFEST_DIR = DATA_ROOT / 'egocentric_tuning_carpenter'
MANIFEST_PATH = MANIFEST_DIR / 'manifest.json'

RUNS = [
    'head_all_spike',
    'head_simple_spike',
    'head_complex_spike',
    'travel_all_spike',
]
SAVE_FORMATS = ['svg']
FIRST_N_MINUTES = 10.0
CATEGORIES = ['CSplus', 'CSminus', 'all-nonPLC']

PLOT_SCRIPT = NOTEBOOKS_HPC / 'run_egocentric_plot_cells.py'
STATS_SCRIPT = NOTEBOOKS_HPC / 'run_egocentric_summary_stats.py'

print(f'[{datetime.now().isoformat(timespec="seconds")}] Local Step-3 run config loaded.')
print(f'Base dir:      {BASE_DIR}')
print(f'Manifest dir:  {MANIFEST_DIR}')
print(f'Runs:          {RUNS}')
print(f'Save formats:  {SAVE_FORMATS}')
print(f'First minutes: {FIRST_N_MINUTES}')


[2026-03-25T14:50:07] Local Step-3 run config loaded.
Base dir:      /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4
Manifest dir:  /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter
Runs:          ['head_all_spike', 'head_simple_spike', 'head_complex_spike', 'travel_all_spike']
Save formats:  ['svg']
First minutes: 10.0


In [2]:
# Preflight: pick a Python interpreter with required packages.
REQUIRED_MODULES = ['numpy', 'pandas', 'matplotlib', 'scipy']

candidate_paths = []
if sys.executable:
    candidate_paths.append(Path(sys.executable).resolve())
candidate_paths.append(Path('/opt/homebrew/Caskroom/miniforge/base/envs/AdamLab/bin/python'))
candidate_paths.append(Path('/opt/homebrew/Caskroom/miniforge/base/envs/adamlab_pipeline/bin/python'))

seen = set()
CANDIDATE_PYTHONS = []
for p in candidate_paths:
    ps = str(p)
    if ps not in seen:
        seen.add(ps)
        CANDIDATE_PYTHONS.append(p)

def check_python(py_path: Path, modules):
    if not py_path.exists() or not os.access(py_path, os.X_OK):
        return False, None, f'not executable: {py_path}'
    check_code = (
        'import importlib, json; '
        f'mods={modules!r}; '
        'out={m:getattr(importlib.import_module(m),"__version__","unknown") for m in mods}; '
        'print(json.dumps(out))'
    )
    res = subprocess.run([str(py_path), '-c', check_code], capture_output=True, text=True)
    if res.returncode != 0:
        return False, None, res.stderr.strip() or res.stdout.strip()
    try:
        versions = json.loads((res.stdout or '').strip())
    except json.JSONDecodeError:
        return False, None, f'non-JSON output: {res.stdout.strip()}'
    return True, versions, ''

SELECTED_PYTHON = None
SELECTED_VERSIONS = None
errors = []

for candidate in CANDIDATE_PYTHONS:
    ok, versions, err = check_python(candidate, REQUIRED_MODULES)
    if ok:
        SELECTED_PYTHON = str(candidate)
        SELECTED_VERSIONS = versions
        break
    errors.append(f'{candidate}: {err}')

if SELECTED_PYTHON is None:
    raise RuntimeError('No usable Python interpreter found for Step 3.\n' + '\n'.join(errors))

print(f'Selected Python: {SELECTED_PYTHON}')
print('Detected versions:')
for mod in REQUIRED_MODULES:
    print(f'  {mod}: {SELECTED_VERSIONS.get(mod)}')


Selected Python: /opt/homebrew/Caskroom/miniforge/base/envs/AdamLab/bin/python3.11
Detected versions:
  numpy: 2.3.5
  pandas: 2.3.3
  matplotlib: 3.10.8
  scipy: 1.16.3


In [3]:
# Status check before cleanup/recompute
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Manifest not found: {MANIFEST_PATH}')

with MANIFEST_PATH.open() as f:
    manifest_rows = json.load(f)
print(f'Manifest rows: {len(manifest_rows)}')

def run_dir(run_name: str) -> Path:
    return MANIFEST_DIR / run_name

def count_files(path: Path, pattern: str, recursive: bool = False) -> int:
    if not path.exists():
        return 0
    return len(list(path.rglob(pattern) if recursive else path.glob(pattern)))

rows = []
for run in RUNS:
    rd = run_dir(run)
    per_cell_results = rd / 'per_cell_results'
    per_cell_summary = rd / 'per_cell_summary'
    summary_stats = rd / 'summary_stats'
    rows.append({
        'run': run,
        'npz': count_files(per_cell_results, 'cell_*.npz'),
        'per_cell_svg': count_files(per_cell_summary, '*.svg', recursive=True),
        'per_cell_png': count_files(per_cell_summary, '*.png', recursive=True),
        'stats_svg': count_files(summary_stats, '*.svg', recursive=True),
        'stats_png': count_files(summary_stats, '*.png', recursive=True),
        'summary_csv': (rd / 'egocentric_tuning_summary.csv').exists(),
    })

header = f"{'run':<20} {'npz':>5} {'per_svg':>8} {'per_png':>8} {'stats_svg':>10} {'stats_png':>10} {'summary_csv':>12}"
print(header)
print('-' * len(header))
for r in rows:
    print(f"{r['run']:<20} {r['npz']:>5} {r['per_cell_svg']:>8} {r['per_cell_png']:>8} {r['stats_svg']:>10} {r['stats_png']:>10} {str(r['summary_csv']):>12}")


Manifest rows: 56
run                    npz  per_svg  per_png  stats_svg  stats_png  summary_csv
-------------------------------------------------------------------------------
head_all_spike          56       46        0          2          0         True
head_simple_spike       56       44        0          2          0         True
head_complex_spike      56       21        0          2          0         True
travel_all_spike        56       47        0          2          0         True


In [4]:
# Full recompute cleanup: remove existing Step-3 outputs for every run.
removed = []
for run in RUNS:
    rd = run_dir(run)
    targets = [
        rd / 'per_cell_summary',
        rd / 'summary_stats',
        rd / 'egocentric_tuning_summary.csv',
    ]
    for t in targets:
        if t.is_dir():
            shutil.rmtree(t)
            removed.append(str(t))
        elif t.is_file():
            t.unlink()
            removed.append(str(t))

print(f'Removed {len(removed)} existing output target(s).')
for p in removed:
    print(f'  removed: {p}')


Removed 12 existing output target(s).
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_all_spike/per_cell_summary
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_all_spike/summary_stats
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_all_spike/egocentric_tuning_summary.csv
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_simple_spike/per_cell_summary
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_simple_spike/summary_stats
  removed: /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/hea

In [5]:
# Step 3a: per-cell summary plots (SVG only)
def run_checked(cmd, cwd=None):
    print('$ ' + ' '.join(str(c) for c in cmd))
    res = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if res.stdout:
        print(res.stdout)
    if res.returncode != 0:
        if res.stderr:
            print(res.stderr)
        raise RuntimeError(f'Command failed with code {res.returncode}: ' + ' '.join(str(c) for c in cmd))
    if res.stderr:
        print('[stderr]')
        print(res.stderr)
    return res

for run in RUNS:
    output_dir = run_dir(run)
    direction_mode = run.split('_')[0]
    cmd = [
        SELECTED_PYTHON,
        str(PLOT_SCRIPT),
        '--output-dir', str(output_dir),
        '--manifest-dir', str(MANIFEST_DIR),
        '--data-root', str(DATA_ROOT),
        '--figures-root', str(FIGURES_ROOT),
        '--direction-mode', direction_mode,
        '--first-n-minutes', str(FIRST_N_MINUTES),
        '--save-formats', *SAVE_FORMATS,
    ]
    print(f'\n=== Step 3a for {run} ===')
    run_checked(cmd, cwd=str(BASE_DIR.parent))

print('\nStep 3a completed for all runs.')



=== Step 3a for head_all_spike ===
$ /opt/homebrew/Caskroom/miniforge/base/envs/AdamLab/bin/python3.11 /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_plot_cells.py --output-dir /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_all_spike --manifest-dir /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter --data-root /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data --figures-root /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures --direction-mode head --first-n-minutes 10.0 --save-formats svg
Loaded manifest: 56 cells
NPZ lookup: 46 success, 10 skipped, 0 missing

--- Loading spatial data ---
Loaded 9 cells from CKII_pAce21_PR_20250806
Loaded 9 cells from CKII_pAce38_PX_20251126
Loaded 15 cells

In [7]:
# Step 3b: summary statistics (SVG only)
for run in RUNS:
    output_dir = run_dir(run)
    cmd = [
        SELECTED_PYTHON,
        str(STATS_SCRIPT),
        '--output-dir', str(output_dir),
        '--manifest-dir', str(MANIFEST_DIR),
        '--categories', *CATEGORIES,
        '--save-formats', *SAVE_FORMATS,
    ]
    print(f'\n=== Step 3b for {run} ===')
    run_checked(cmd, cwd=str(BASE_DIR.parent))

print('\nStep 3b completed for all runs.')



=== Step 3b for head_all_spike ===
$ /opt/homebrew/Caskroom/miniforge/base/envs/AdamLab/bin/python3.11 /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/notebooks_HPC/run_egocentric_summary_stats.py --output-dir /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter/head_all_spike --manifest-dir /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/egocentric_tuning_carpenter --categories CSplus CSminus all-nonPLC --save-formats svg
Loaded manifest: 56 cells
NPZ results: 46 success (in requested categories), 10 skipped, 0 missing
Total rows for analysis: 46

=== Pass Counts ===
            pass_95  pass_99  pass_100  n_cells
category                                       
CSplus            7        5         3       12
CSminus           2        0         0        8
all-nonPLC       11        7         5       26

=== MRL Statistics ===
          

In [8]:
# Final validation report
required_stats_csv = {'pass_counts.csv', 'mrl_stats.csv', 'pairwise_mannwhitney.csv'}
failures = []
report_rows = []

for run in RUNS:
    rd = run_dir(run)
    per_cell_root = rd / 'per_cell_summary'
    stats_root = rd / 'summary_stats'
    summary_csv = rd / 'egocentric_tuning_summary.csv'

    per_cell_svg = len(list(per_cell_root.rglob('*.svg'))) if per_cell_root.exists() else 0
    per_cell_png = len(list(per_cell_root.rglob('*.png'))) if per_cell_root.exists() else 0
    stats_svg = len(list(stats_root.rglob('*.svg'))) if stats_root.exists() else 0
    stats_png = len(list(stats_root.rglob('*.png'))) if stats_root.exists() else 0
    stats_csv = sorted(p.name for p in stats_root.glob('*.csv')) if stats_root.exists() else []
    missing_csv = sorted(required_stats_csv - set(stats_csv))

    report_rows.append({
        'run': run,
        'per_cell_svg': per_cell_svg,
        'per_cell_png': per_cell_png,
        'stats_svg': stats_svg,
        'stats_png': stats_png,
        'stats_csv': ', '.join(stats_csv),
        'summary_csv': summary_csv.exists(),
    })

    if per_cell_svg == 0:
        failures.append(f'{run}: no per-cell SVG files found')
    if stats_svg == 0:
        failures.append(f'{run}: no summary-stats SVG files found')
    if per_cell_png > 0 or stats_png > 0:
        failures.append(f'{run}: PNG outputs found (expected SVG-only)')
    if missing_csv:
        failures.append(f"{run}: missing stats CSV(s): {', '.join(missing_csv)}")
    if not summary_csv.exists():
        failures.append(f'{run}: missing egocentric_tuning_summary.csv')

header = f"{'run':<20} {'per_svg':>8} {'per_png':>8} {'stats_svg':>10} {'stats_png':>10} {'summary_csv':>12}"
print(header)
print('-' * len(header))
for r in report_rows:
    print(f"{r['run']:<20} {r['per_cell_svg']:>8} {r['per_cell_png']:>8} {r['stats_svg']:>10} {r['stats_png']:>10} {str(r['summary_csv']):>12}")

if failures:
    raise AssertionError('Validation failed:\n - ' + '\n - '.join(failures))

print('\nValidation passed for all runs. Outputs are SVG-only with required CSV artifacts.')


run                   per_svg  per_png  stats_svg  stats_png  summary_csv
-------------------------------------------------------------------------
head_all_spike             46        0          2          0         True
head_simple_spike          44        0          2          0         True
head_complex_spike         21        0          2          0         True
travel_all_spike           47        0          2          0         True

Validation passed for all runs. Outputs are SVG-only with required CSV artifacts.
